# vgra-yolo: Part A evaluation session (I1 + I3 + I4 + I2)

One notebook for the evaluation of every trained detector (docs/PROTOCOL.md, Part A).

**Two modes, set in the first code cell:**
- `OPEN_TEST = False` (**dry run**, default): validation split only. Checks the whole pipeline on Kaggle
  with the detectors that exist. Safe to run any time; the test split is never read.
- `OPEN_TEST = True` (**the one test session**): only when every detector in `RUNS` has FINISHED and the code
  is tagged. Runs predictions on valid and test (I1), the RQ1 premise test on the test pairs (I3), standard
  Ultralytics validation on test (I4) and the patient-level analysis (I2). The test split is opened once:
  if a bug is found afterwards, it is fixed, everything is rerun, and the event is logged (protocol section 2).

Settings: GPU **T4 x1** is enough (about 1-1.5 h for 8 detectors), Internet **on**.
Inputs: the images, `vgra-index`, and the trained runs (`runs/<name>/weights/best.pt`). For the test session: the one
dataset `vgra-final-detectors` (the 8 frozen runs); every best.pt must match the SHA-256 in `docs/final_weights.json`.
The output of this notebook (`/kaggle/working/runs/...`) holds every table; download it or make it a dataset.
Every step runs through `sh()`, which stops the notebook if the step fails, so no step runs on missing files.

In [ ]:
# Settings
REPO = "https://github.com/nithoukhan1/vgra-yolo.git"
REF = "main"        # dry run: main is fine; the test session: a fixed tag made just before (e.g. "parta-test-v1")
OPEN_TEST = False   # True ONLY for the one test session
RUNS = ["B0_s0", "B1_s0"]  # test session: ["B0_s0", "B0_s1", "B0_s2", "B1_s0", "V8_s0", "V8_s1", "V8R_s0", "V8R_s1"]
# the test session attaches ONE dataset with the frozen runs (vgra-final-detectors), whose best.pt files must
# match docs/final_weights.json
RUN_ARGS = " ".join(RUNS)
READY = False       # set True by the check cell; every later cell refuses to run without it

In [ ]:
# Clone the fixed code, install, check
%cd /kaggle/working
!rm -rf vgra-yolo
!git clone --quiet {REPO} vgra-yolo
%cd /kaggle/working/vgra-yolo
!git checkout --quiet {REF}
!git log -1 --oneline
!pip install -q -r requirements.txt
!pip install -q -e .
!python scripts/00_check_setup.py

In [ ]:
# Every run must be found (and FINISHED for the test session) before anything is evaluated
import hashlib
import json
import subprocess
import sys
sys.path.insert(0, "/kaggle/working/vgra-yolo")
from vgra.runs import find_previous_run, is_finished, read_metrics


def sh(cmd):
    """Run one step, show its output, and stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd="/kaggle/working/vgra-yolo", stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"step failed (exit code {proc.returncode}): {cmd}")


FROZEN = json.load(open("/kaggle/working/vgra-yolo/docs/final_weights.json"))  # SHA-256 of the frozen best.pt files


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


problems, warnings = [], []
for r in RUNS:
    p = find_previous_run(r, "/kaggle/input")
    if p is None or not (p / "weights" / "best.pt").is_file():
        problems.append(f"{r}: not found under /kaggle/input (attach its dataset)")
        continue
    v = read_metrics(p).get("valid", {})
    print(f"{r}: {p} | finished {is_finished(p)} | valid mAP50 {v.get('map50')} | mAP50-95 {v.get('map50_95')}")
    if not is_finished(p):
        (problems if OPEN_TEST else warnings).append(f"{r}: not finished")
    digest = sha256(p / "weights" / "best.pt")
    frozen = FROZEN.get(r, {}).get("sha256")
    print(f"    best.pt sha256 {digest[:16]}... " + ("matches docs/final_weights.json" if digest == frozen
          else "DOES NOT MATCH docs/final_weights.json" if frozen else "not in docs/final_weights.json"))
    if digest != frozen:
        (problems if OPEN_TEST else warnings).append(f"{r}: best.pt is not the frozen file")
if problems:
    raise RuntimeError("STOP: " + "; ".join(problems))  # a real error, so Run All stops here
for w in warnings:
    print(f"WARNING (dry run only): {w}; the test session needs the finished run")
print("\nOPEN_TEST is True: this session opens the test split (protocol section 2)." if OPEN_TEST
      else "\nDry run: validation split only.")
READY = True

In [ ]:
# I1: predictions on valid (every run), then on test (test session only)
assert READY, "the check cell did not pass; fix it and run again"
sh(f"python scripts/10_predict.py --runs {RUN_ARGS} --split valid")
if OPEN_TEST:
    sh(f"python scripts/10_predict.py --runs {RUN_ARGS} --split test --open-test")

In [ ]:
# I3: RQ1 premise test on the test pairs (test session only; on valid it was Gate 2, already done).
# Dry run: a short version on 64 valid pairs, only to prove the step works.
assert READY, "the check cell did not pass; fix it and run again"
if OPEN_TEST:
    sh(f"python scripts/04_premise_test.py --runs {RUN_ARGS} --split test --open-test")
else:
    sh(f"python scripts/04_premise_test.py --runs {RUN_ARGS} --split valid --limit 64 --boot 50 --out /kaggle/working/dry")

In [ ]:
# I4: standard Ultralytics validation (detector table T2)
assert READY, "the check cell did not pass; fix it and run again"
split = "test --open-test" if OPEN_TEST else "valid"
sh(f"python scripts/11_val_detectors.py --runs {RUN_ARGS} --split {split}")

In [ ]:
# I2: patient-level analysis (CPU), thresholds from valid
assert READY, "the check cell did not pass; fix it and run again"
split = "test" if OPEN_TEST else "valid"
sh(f"python scripts/12_study_level.py --pred /kaggle/working/runs/predictions --runs {RUN_ARGS} --eval {split}")

In [ ]:
# What was produced
sh(r'find /kaggle/working/runs -maxdepth 4 \( -name "*.csv" -o -name "*.json" -o -name "summary.txt" \) -not -path "*/lists/*" | sort')